# Solution two : 2 Feature Selection + data augmentation + Train + log to MLFlow

# 0.Library

In [5]:
# general
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import re
import importlib

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_validate
from sklearn.pipeline import Pipeline

# MLFlow
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

# Paths
import sys
sys.path.append('../') 
from features import data_utils as du
from features import data_pipeline as dp
from features import general_func as gf
import constants_data as cd
from features import model_utils as mu
import paths_data as paths

# Reload
importlib.reload(du)
importlib.reload(dp)
importlib.reload(gf)
importlib.reload(cd)
importlib.reload(mu)
importlib.reload(paths)

<module 'paths_data' from '/home/smira/myproject/detection_AD_with_VR_data/src/notebooks/../paths_data.py'>

# 1. Paths and Constants

In [6]:
# Read dataframe
models_paths_folder = paths.MODELS_DIR_SOL_TWO

# Read dataframe input (train set)
name_input_df = "cleaned_features_13_patients.csv"
df_path = paths.STAGE_THREE_DATA_PREPROCESSING/ name_input_df
df      = pd.read_csv(df_path)
df.head()

,Age,Help_Rating_out_of_5,MoCA_Score,Tutorial_total_reading_time,Tutorial_intensity_reading_time,Tutorial_total_count_hover,Tutorial_total_duration_hover,Tutorial_mean_duration_hover,Tutorial_max_duration_hover,Tutorial_total_count_press,...,Memory_Yaw_std,Memory_Pitch_std,Memory_Roll_std,Memory_Yaw_range,Memory_Roll_range,Memory_dominant_hand_mean_speed,Memory_not_dominant_hand_mean_speed,Memory_dominant_hand_z_range,Memory_not_dominant_hand_y_range,Gender_Male
0,73.0,4,28.0,45.03,0.92,32,20.64,1.03,2.29,8,...,26.23,161.72,177.36,99.05,360.00,0.16,0.03,0.82,0.80,0
1,82.0,1,26.0,338.75,0.99,688,145.79,2.11,20.01,9,...,23.27,161.90,162.61,109.38,359.99,0.06,0.01,0.98,0.75,1
2,62.0,1,27.0,152.90,0.97,61,54.33,1.81,11.62,8,...,30.07,170.64,169.17,248.93,359.98,0.12,0.02,1.05,0.58,1
3,78.0,2,27.0,181.69,0.98,26,91.62,2.78,23.45,9,...,31.96,136.51,170.83,174.64,359.98,0.12,0.02,0.82,0.42,0
4,71.0,2,26.0,190.51,0.98,109,102.68,2.57,21.86,8,...,27.48,169.32,125.66,146.78,359.99,0.11,0.03,0.82,0.64,0


# 3. Data Splitting

In [7]:
y = df['MoCA_Score']
df = df.drop(columns=['MoCA_Score'])
X = df

# 4. Train and Test

In [8]:
cd.all_experiments_sol_2

[('gaussian_noise',
  [('LinearRegression', sklearn.linear_model._base.LinearRegression),
   ('Ridge', sklearn.linear_model._ridge.Ridge),
   ('RandomForest', <function constants_data.<lambda>()>),
   ('XGBRegressor', <function constants_data.<lambda>()>),
   ('SVR', sklearn.svm._classes.SVR),
   ('Lasso', sklearn.linear_model._coordinate_descent.Lasso)]),
 ('smoter',
  [('LinearRegression', sklearn.linear_model._base.LinearRegression),
   ('Ridge', sklearn.linear_model._ridge.Ridge),
   ('RandomForest', <function constants_data.<lambda>()>),
   ('XGBRegressor', <function constants_data.<lambda>()>),
   ('SVR', sklearn.svm._classes.SVR),
   ('Lasso', sklearn.linear_model._coordinate_descent.Lasso)]),
 ('gaussian_copula',
  [('LinearRegression', sklearn.linear_model._base.LinearRegression),
   ('Ridge', sklearn.linear_model._ridge.Ridge),
   ('RandomForest', <function constants_data.<lambda>()>),
   ('XGBRegressor', <function constants_data.<lambda>()>),
   ('SVR', sklearn.svm._classes.

In [9]:
def rescale_func(X_train, X_test):
    from sklearn.preprocessing import StandardScaler

    # create transformer
    scaler = StandardScaler()
    scaler.fit(X_train)

    return scaler.transform(X_train), scaler.transform(X_test)

In [23]:
original_cols = X.columns.to_list()
original_cols

['Age',
 'Help_Rating_out_of_5',
 'Tutorial_total_reading_time',
 'Tutorial_intensity_reading_time',
 'Tutorial_total_count_hover',
 'Tutorial_total_duration_hover',
 'Tutorial_mean_duration_hover',
 'Tutorial_max_duration_hover',
 'Tutorial_total_count_press',
 'Tutorial_median_duration_press',
 'Tutorial_total_duration_grab',
 'Tutorial_total_count_gaze',
 'Tutorial_max_duration_gaze',
 'Tutorial_median_duration_gaze',
 'Tutorial_intensity_gaze',
 'Tutorial_decision_latency',
 'Tutorial_head_total_distance',
 'Tutorial_HMD_Y_std',
 'Tutorial_HMD_Z_std',
 'Tutorial_HMD_Y_range',
 'Tutorial_head_total_orientation',
 'Tutorial_Yaw_std',
 'Tutorial_Roll_std',
 'Tutorial_Pitch_range',
 'Tutorial_Roll_range',
 'Tutorial_max_head_speed_in_orientation',
 'Tutorial_not_dominant_hand_mean_speed',
 'Tutorial_dominant_hand_max_speed',
 'Tutorial_dominant_hand_x_range',
 'Tutorial_not_dominant_hand_x_range',
 'Tutorial_dominant_hand_trigger_press_count',
 'Tutorial_dominant_hand_trigger_pressure_

In [24]:
X_train =X.iloc[:11]
y_train = y.iloc[:11]

X_test =X.iloc[[-1]]
y_test = y.iloc[[-1]]

In [21]:
type(X_test)

pandas.core.frame.DataFrame

In [25]:
X_train_scaled, X_test_scaled = rescale_func(X_train, X_test)

In [30]:
X_new, y_new = dp.guassian_noise_pipeline(mean = 0.0, standard_deviation=1.0, 
                            X_train_scaled= X_train_scaled, y_train= y_train,
                            original_cols= X.columns.to_list())

In [34]:
X_train.shape

(11, 112)

In [33]:
y_new.shape

(22,)

In [31]:
X_new.shape

(22, 112)

In [35]:
X_new

,Age,Help_Rating_out_of_5,Tutorial_total_reading_time,Tutorial_intensity_reading_time,Tutorial_total_count_hover,Tutorial_total_duration_hover,Tutorial_mean_duration_hover,Tutorial_max_duration_hover,Tutorial_total_count_press,Tutorial_median_duration_press,...,Memory_Yaw_std,Memory_Pitch_std,Memory_Roll_std,Memory_Yaw_range,Memory_Roll_range,Memory_dominant_hand_mean_speed,Memory_not_dominant_hand_mean_speed,Memory_dominant_hand_z_range,Memory_not_dominant_hand_y_range,Gender_Male
0,-0.074387,1.917029,-1.576088,-2.996604,-0.472173,-1.664867,-1.871021,-1.639980,-0.693375,-0.860170,...,-0.560862,0.128516,0.955211,-1.086911,1.206241,0.951842,0.912871,-0.880177,1.420797,-0.755929
1,1.398475,-1.095445,0.888014,0.665912,3.022394,1.496167,-0.401905,0.164100,0.832050,-0.413855,...,-0.787294,0.146525,0.494884,-0.988323,0.748701,-1.665723,-1.095445,-0.064855,1.130299,1.322876
2,-1.874552,-1.095445,-0.671136,-0.380521,-0.317688,-0.813926,-0.809992,-0.690089,-0.693375,-0.279961,...,-0.267114,1.020944,0.699612,0.343528,0.291162,-0.095184,-0.091287,0.291848,0.142608,1.322876
3,0.743870,-0.091287,-0.429608,0.142695,-0.504136,0.127943,0.509491,0.514328,0.832050,0.077091,...,-0.122534,-2.393693,0.751418,-0.365488,0.291162,-0.095184,-0.091287,-0.880177,-0.786984,-0.755929
4,-0.401690,-0.091287,-0.355614,0.142695,-0.061988,0.407296,0.223830,0.352449,-0.693375,-0.146067,...,-0.465241,0.888881,-0.658275,-0.631381,0.748701,-0.356941,0.912871,-0.880177,0.491205,-0.755929
5,-0.728992,-1.095445,-0.204607,0.142695,-0.562734,0.641943,1.801770,0.988764,-0.693375,-0.592381,...,-0.777349,-0.748905,0.957707,-1.040051,-1.081457,-1.403966,0.912871,-1.644541,-1.309879,-0.755929
6,1.234824,-0.091287,0.276267,0.665912,-0.344323,0.347435,1.271255,-0.040539,2.357476,-0.146067,...,-0.126359,0.363629,-0.369595,1.368448,-0.623918,0.166572,-1.095445,0.903339,0.433105,-0.755929
7,0.089264,-0.091287,-0.189255,0.142695,-0.397594,-0.797761,-0.687566,-0.738958,-0.693375,-0.235329,...,0.462668,-1.084065,0.615349,1.318820,0.748701,0.951842,-1.095445,-0.319643,-1.309879,-0.755929
8,-1.383598,-0.091287,-0.575666,0.142695,-0.365632,-0.665410,-0.116243,-0.539410,-0.693375,-0.369224,...,2.978656,-0.040565,0.211509,1.377419,-0.166378,0.690085,-0.091287,0.393763,-0.554586,1.322876
9,0.907521,1.917029,2.526610,0.665912,0.529318,1.707323,0.754344,2.276665,0.832050,3.067398,...,-0.175317,1.021945,-2.135691,0.734256,0.291162,-0.880453,-1.095445,1.361958,1.362697,1.322876


In [ ]:
# experiment list
experiments_list = cd.all_experiments_sol_2

for exp_name, models_list in experiments_list:
    
    print(f"======== Experiment: {exp_name} ===========")
    mlflow.set_experiment(f"MoCA_Regression_data_augmentation_{exp_name}")

    for model in models_list:
        print(f"================ Run: {exp_name}_{model.__name__} =================")
        with mlflow.start_run(run_name=f"{exp_name}_{model.__name__}"):

            pipe = mu.make_model_pipeline(
                mu.make_feature_pipeline(exp_name, k_value),
                model()
            )

            scores = cross_validate(
                pipe,
                X,
                y,
                cv=5,
                scoring={
                    "r2": "r2",
                    "mae": "neg_mean_absolute_error",
                    "mse": "neg_mean_squared_error",
                    "rmse": "neg_root_mean_squared_error"
                },
                return_train_score=True
            )

            # compute metrics of test set
            r2_test   = scores["test_r2"].mean()
            mae_test  = -scores["test_mae"].mean()
            mse_test  = -scores["test_mse"].mean()
            rmse_test = -scores["test_rmse"].mean()

            # compute metrics of train set
            r2_train   = scores["train_r2"].mean()
            mae_train  = -scores["train_mae"].mean()
            mse_train  = -scores["train_mse"].mean()
            rmse_train = -scores["train_rmse"].mean()

            # print
            print(f"Model: {model.__name__}")
            print(f"R² of test: {mse_test:.3f}")

            # MLflow logging
            mlflow.log_param("experiment", exp_name)
            mlflow.log_param("model", model.__name__)

            mlflow.log_metric("r2_test", r2_test)
            mlflow.log_metric("mae_test", mae_test)
            mlflow.log_metric("mse_test", mse_test)
            mlflow.log_metric("rmse_test", rmse_test)

            mlflow.log_metric("r2_test", r2_train)
            mlflow.log_metric("mae_test", mae_train)
            mlflow.log_metric("mse_test", mse_train)
            mlflow.log_metric("rmse_test", rmse_train)

            mlflow.log_metric("gap", abs(mse_test - mse_train))

            # log model
            mlflow.sklearn.log_model(pipe, name="model", skops_trusted_types=["xgboost.sklearn.XGBRegressor", "sklearn.feature_selection._univariate_selection.f_regression"])


## 5.0 Split train/test and Scale

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)

In [ ]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5.1 Gaussian noise

In [ ]:
X_train_aug , y_train_aug = du.guassian_noise_pipeline(loc=0, scale=1, 
                                                       X_train_scaled= X_train_scaled, y_train= y_train, 
                                                       original_cols= X_train.columns)

In [ ]:
X_train_aug.shape , y_train_aug.shape

## 5.2 SMOTER

In [ ]:
import numpy as np

Q1 = np.quantile(y_train, 0.25)
Q2 = np.quantile(y_train, 0.50)  # median
Q3 = np.quantile(y_train, 0.75)

print("Q1:", Q1)
print("Q2:", Q2)
print("Q3:", Q3)

IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print("lower_fence:",lower_fence)
print("upper_fence:", upper_fence)